In [1]:
!pip install -U langgraph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.2/250.2 kB 6.0 MB/s eta 0:00:00
  Attempting uninstall: langgraph
    Found existing installation: langgraph 1.2.11
    Uninstalling langgraph-1.2.11:
      Successfully uninstalled langgraph-1.2.11


In [2]:
!pip install -U langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 3.9 MB/s eta 0:00:00


In [21]:
from langgraph.graph import StateGraph ,START,END
from typing import Annotated ,TypedDict
from langchain_core.messages import BaseMessage,HumanMessage
from langchain_groq import ChatGroq
from langgraph.checkpoint.memory import MemorySaver

In [5]:
from google.colab import userdata
groq_api_key=userdata.get('GROQ_API_KEY')

In [6]:
llm=ChatGroq(model="openai/gpt-oss-20b",api_key=groq_api_key)

In [7]:
from langgraph.graph.message import add_messages
class ChatState(TypedDict):
  messages:Annotated[list[BaseMessage],add_messages]


In [8]:
def chat_node(state:ChatState):
  messages=state['messages']
  response=llm.invoke(messages)
  return {
      'messages':[response]
  }

In [23]:
checkpoint=MemorySaver()
graph=StateGraph(ChatState)
graph.add_node('chat_node',chat_node)
graph.add_edge(START,'chat_node')
graph.add_edge('chat_node',END)
chatbot=graph.compile(checkpointer=checkpoint)


In [31]:
thread_id_1="1"
initial_state={
    'messages':[HumanMessage(content="What is my name")]
}

In [32]:
config={'configurable':{'thread_id':thread_id_1}}

response=chatbot.invoke(initial_state,config=config)['messages'][-1].content

In [33]:
print(response)

Your name is Lakshay.


In [28]:
thread_id_1="1"
thread_id_2="2"

In [30]:
while True:
  user_message=input("Type_here...")
  print('User',user_message)
  if user_message.strip().lower() in ['exit','quit','bye']:
    break
  config={'configurable':{'thread_id':thread_id_1}}
  response=chatbot.invoke({'messages':[HumanMessage(content=user_message)]},config=config)
  print('AI',response['messages'][-1].content)

Type_here...hi i am lakshay
User hi i am lakshay
AI Hello Lakshay! 👋 How can I assist you today?
Type_here...what is generative ai
User what is generative ai
AI **Generative AI** is a class of artificial‑intelligence systems that can *create* new content—text, images, audio, video, code, or even 3D models—rather than just analyze or classify existing data.  

---

### Core Idea
- **Generative models learn a statistical description of a data distribution** (e.g., the way language is structured, or the visual patterns in photos).  
- Once trained, they can **sample from that distribution** to produce novel examples that resemble the training data but are not exact copies.

### Common Generative Architectures
| Architecture | How it works | Typical use |
|--------------|--------------|-------------|
| **Variational Autoencoders (VAEs)** | Encode data to a low‑dimensional latent space, then decode to reconstruct. Sampling from latent space yields new data. | Image synthesis, style transfer

KeyboardInterrupt: Interrupted by user